# The Mathematics of Organizational Network Analysis
## Interactive Companion Notebook, v3

This notebook is a hands-on companion to the essay *The Mathematics of Organizational Network Analysis*. Every major concept is illustrated with working Python code, concrete numerical examples, and visualizations. Sections follow the essay in order.

**How to use this notebook:** Run cells top-to-bottom. Markdown headers introduce the essay concept; code cells demonstrate it.

---

### Table of Contents
| Part | Section |
|------|---------|
| 0 | [Setup & File-Size Configuration](#setup) |
| 1 | [The Adjacency Matrix](#adjacency) |
| 2 | [Degree Matrix & Laplacian L = D − A](#laplacian) |
| 3 | [Laplacian Intuition: Heat Diffusion Analogy](#heat) |
| 4 | [Fiedler Value & Algebraic Connectivity](#fiedler) |
| 4b | [Interlude: Sign Ambiguity and Eigenvalue Degeneracy](#sign-degeneracy) |
| 5 | [Spectral Clustering: Finding Informal Communities](#spectral-clustering) |
| 6 | [Spectral Graph Drawing: Eigenvectors as Geometry](#spectral-drawing) |
| 8 | [Directed Networks: In/Out Centrality](#directed) |
| 9 | [Centrality: Degree, Eigenvector, Betweenness, PageRank](#centrality) |
| 10 | [Matrix Powers: Walks, Triangles & Structural Holes](#powers) |
| 11 | [Normalised Laplacian & Random Walk Mixing](#random-walk) |
| 12 | [Cospectral Graphs: Limits of Spectral Fingerprints](#cospectral) |
| 13 | [Bipartite Networks: People Connecting Through Things](#bipartite) |
| 14 | [Hypergraph Laplacian: Group Interactions](#hypergraph) |
| 15 | [Multi-Layer Networks & Supra-Laplacian](#multilayer) |
| 16 | [Temporal Tracking: Spectral Health Dashboard](#temporal) |
| 17 | [Data Quality: Threshold Sensitivity](#threshold) |
| 18 | [Summary Reference Table](#reference) |

*Note: Part 7 (Spectral Graph Drawing) was renumbered to Part 6; no content was removed.*

<!-- cross-links -->
### 📄 Read the essay · 💻 GitHub

| Resource | Link |
|----------|------|
| Essay (rendered) | [https://johanm.github.io/graph-matrixmath-ona/graphmatrixmath.html](https://johanm.github.io/graph-matrixmath-ona/graphmatrixmath.html) |
| GitHub repository | [https://github.com/johanm/graph-matrixmath-ona](https://github.com/johanm/graph-matrixmath-ona) |
| Open this notebook in Colab | [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johanm/graph-matrixmath-ona/blob/main/graphmatrixmath.ipynb) |

---


<a id='setup'></a>
---
## Part 0: Setup & File-Size Configuration

**Why this cell exists:** Jupyter embeds every plot as a base64-encoded image inside the `.ipynb` JSON file. At default 120 DPI, a notebook with ~30 figures can balloon to 30–60 MB, triggering a `413 Request Entity Too Large` error on save.

The fixes applied here:
- **SVG output format**: SVG is text-based XML, not a raster bitmap. File sizes are typically 10–50× smaller than PNG for network diagrams.
- **Reduced DPI (72)**: only used as a fallback for raster output; SVG ignores DPI.
- **`plt.close(fig)` after every plot**: releases figure memory from the kernel and prevents duplicate rendering in the output cell.
- **`bbox_inches='tight'`**: trims whitespace, slightly reducing output size.

Run this cell first every session.

In [ ]:
# ── Core dependencies ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from scipy.linalg import eigh, svd
from sklearn.cluster import KMeans
from itertools import permutations
import warnings
warnings.filterwarnings('ignore')

# ── FILE-SIZE FIX: use SVG output instead of PNG ───────────────────────────────
# SVG is text-based, so embedded figures are ~10-50x smaller than PNG.
# This is the primary fix for the 413 'Request Entity Too Large' save error.
%config InlineBackend.figure_formats = ['svg']

# ── Shared plot style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 72,            # lower DPI = smaller raster fallback
    'savefig.bbox': 'tight',     # trim whitespace
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
})

NODE_COLOR = '#4C72B0'
EDGE_COLOR = '#AAAAAA'
ACCENT1    = '#DD8452'
ACCENT2    = '#55A868'
ACCENT3    = '#C44E52'

# ── Shared helper functions ────────────────────────────────────────────────────
def fiedler_value(G):
    """Return λ₂ (Fiedler value) of the graph Laplacian."""
    L = nx.laplacian_matrix(G).toarray().astype(float)
    return np.sort(np.linalg.eigvalsh(L))[1]

def normalized_laplacian_spectrum(G):
    """Return sorted eigenvalues of the symmetric normalised Laplacian."""
    L = nx.normalized_laplacian_matrix(G).toarray().astype(float)
    return np.sort(np.linalg.eigvalsh(L))

def laplacian(A):
    """Build Laplacian from a numpy adjacency matrix."""
    return np.diag(A.sum(axis=1)) - A

print('Setup complete. SVG output enabled — notebook will stay small on save.')

<a id='adjacency'></a>
---
## Part 1: The Adjacency Matrix

> *The adjacency matrix A is an n × n table where each cell A_ij equals 1 if person i and person j have a connection … For undirected networks the matrix is symmetric: A_ij always equals A_ji. The row sum of row i gives the degree of person i, that is, their number of direct connections.*

We build the exact five-person example from the essay: Ana, Bo, Cy, Dev, Eve, with edges A–B, A–C, B–C, B–D, C–E.

In [ ]:
# ── Five-person essay example ──────────────────────────────────────────────────
nodes = ['Ana', 'Bo', 'Cy', 'Dev', 'Eve']
edges = [('Ana','Bo'), ('Ana','Cy'), ('Bo','Cy'), ('Bo','Dev'), ('Cy','Eve')]

G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)

n   = len(nodes)
idx = {name: i for i, name in enumerate(nodes)}
A   = np.zeros((n, n), dtype=int)
for u, v in edges:
    A[idx[u], idx[v]] = 1
    A[idx[v], idx[u]] = 1

print('Adjacency matrix A:')
print(pd.DataFrame(A, index=nodes, columns=nodes).to_string())
print(f'\nSymmetric (undirected): {np.array_equal(A, A.T)}')
print('Node degrees (row sums):', dict(zip(nodes, A.sum(axis=1))))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

pos = nx.spring_layout(G, seed=42)
deg = dict(G.degree())
nx.draw_networkx(G, pos, ax=ax1,
                 node_size=[600 + deg[v]*300 for v in G.nodes()],
                 node_color=NODE_COLOR, edge_color=EDGE_COLOR,
                 font_color='white', font_weight='bold', width=2)
ax1.set_title('Network graph\n(node size ∝ degree)', fontsize=12)
ax1.axis('off')

ax2.imshow(A, cmap='Blues', vmin=0, vmax=1.5)
ax2.set_xticks(range(n)); ax2.set_yticks(range(n))
ax2.set_xticklabels(nodes); ax2.set_yticklabels(nodes)
ax2.set_title('Adjacency matrix A\n(blue cell = connected pair)', fontsize=12)
for i in range(n):
    for j in range(n):
        ax2.text(j, i, A[i,j], ha='center', va='center', fontsize=13,
                 color='white' if A[i,j] else '#888')

plt.suptitle('Network and its adjacency matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
plt.close(fig)

<a id='laplacian'></a>
---
## Part 2: The Degree Matrix and the Laplacian L = D − A

> *The Laplacian matrix L = D − A is the most analytically important object in graph theory. Every row of L sums to zero. The number of zero eigenvalues equals the number of disconnected components.*

We construct D and L step by step, verify every property, and compute the eigenvalues and Fiedler vector values.

In [ ]:
D_mat = np.diag(A.sum(axis=1))
L_mat = D_mat - A

print('Degree matrix D:')
print(pd.DataFrame(D_mat, index=nodes, columns=nodes).to_string())
print('\nLaplacian L = D − A:')
print(pd.DataFrame(L_mat, index=nodes, columns=nodes).to_string())
print('\nRow sums of L (must all be 0):', L_mat.sum(axis=1).tolist())

In [ ]:
eigenvalues, eigenvectors = eigh(L_mat)

print('Spectrum of L:', [f'{v:.4f}' for v in eigenvalues])
print(f'\nλ₁ = {eigenvalues[0]:.6f}  (0 → network is connected)')
print(f'λ₂ = {eigenvalues[1]:.4f}  (Fiedler value — algebraic connectivity)')

fv = eigenvectors[:, 1]
# Robust sign fix: orient so the component with largest absolute value is positive
fv = fv * np.sign(fv[np.argmax(np.abs(fv))])

# Correct values: Ana≈0.00, Bo≈−0.205, Cy≈+0.205, Dev≈−0.677, Eve≈+0.677
# The Fiedler split is {Bo, Dev} (negative) vs {Cy, Eve} (positive),
# with Ana sitting exactly at zero — the geometric midpoint of the two arms.
# Note: Ana is NOT an articulation point; Bo and Cy are (removing either disconnects the graph).
print('\nFiedler vector:')
for name, val in zip(nodes, fv):
    print(f'  {name}: {val:+.4f}')
print('\nStructural reading: Bo−Dev arm (negative) | Ana (midpoint, ≈0) | Cy−Eve arm (positive)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = [ACCENT3 if i == 1 else NODE_COLOR for i in range(n)]
ax1.bar(range(1, n+1), eigenvalues, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_xticks(range(1, n+1))
ax1.set_xticklabels([f'λ{i}' for i in range(1, n+1)], fontsize=11)
ax1.set_ylabel('Eigenvalue'); ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_title('Laplacian spectrum\n(red bar = Fiedler value λ₂)', fontsize=12)
for i, v in enumerate(eigenvalues):
    ax1.text(i+1, v+0.05, f'{v:.2f}', ha='center', fontsize=9)

bar_colors = [ACCENT1 if fv[i] > 0 else NODE_COLOR for i in range(n)]
ax2.barh(nodes, fv, color=bar_colors, edgecolor='white')
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('Fiedler vector value')
ax2.set_title('Fiedler vector: natural two-way split\n(orange = periphery, blue = core)', fontsize=12)
for i, (name, val) in enumerate(zip(nodes, fv)):
    ax2.text(val + (0.01 if val >= 0 else -0.01), i, f'{val:+.3f}',
             va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.suptitle('Laplacian eigenstructure', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

### The three Laplacian variants: combinatorial, weighted, and normalized

> *The diagonal-equals-degree, off-diagonal-equals-−1 pattern holds exactly for the unweighted combinatorial Laplacian. Weighted and normalized variants modify it in principled, predictable ways.*

The essay explains the three variants in detail. Here we demonstrate all three on the same five-node network so the differences are visible side by side, and confirm that all three preserve the essential Laplacian properties (rows sum to zero, non-negative eigenvalues, zero eigenvalue count = component count).


In [ ]:
# ── Three Laplacian variants on the five-node essay network ──────────────────
#
# 1. Combinatorial (unweighted): diagonal = degree, off-diag = −1 or 0
# 2. Weighted:                   diagonal = sum of weights, off-diag = −w_ij
# 3. Normalized:                 diagonal = 1, off-diag = −1/√(d_i · d_j)
#
# All three: rows sum to zero, #zero eigenvalues = #components (≥1 always).

# ── 1. Combinatorial Laplacian (already computed as L_mat) ────────────────────
L_comb = L_mat.astype(float).copy()

# ── 2. Weighted Laplacian ─────────────────────────────────────────────────────
# Assign realistic tie strengths (e.g. weekly interaction frequency 1-5)
edge_weights = {('Ana','Bo'): 4, ('Ana','Cy'): 2,
                ('Bo','Cy'): 5, ('Bo','Dev'): 1, ('Cy','Eve'): 3}

W_mat = np.zeros((n, n))
for (u, v), w in edge_weights.items():
    i_, j_ = idx[u], idx[v]
    W_mat[i_, j_] = W_mat[j_, i_] = w

D_wt  = np.diag(W_mat.sum(axis=1))
L_wt  = D_wt - W_mat

# ── 3. Normalized Laplacian: D^{-1/2} L D^{-1/2} ────────────────────────────
degree_vec = A.sum(axis=1).astype(float)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degree_vec))   # D^{-1/2}
L_norm_mat = D_inv_sqrt @ L_comb @ D_inv_sqrt

print('── 1. Combinatorial Laplacian L = D − A ──────────────────────────────────')
print(pd.DataFrame(L_comb, index=nodes, columns=nodes).round(3).to_string())
print(f'  Diagonal = degrees: {np.diag(L_comb).astype(int).tolist()}')
print(f'  Off-diag values (edges): {-1}  |  Row sums: {L_comb.sum(axis=1).tolist()}')

print()
print('── 2. Weighted Laplacian ──────────────────────────────────────────────────')
print(pd.DataFrame(L_wt, index=nodes, columns=nodes).round(3).to_string())
print(f'  Diagonal = weighted degrees (sum of incident weights): {np.diag(L_wt).astype(int).tolist()}')
print(f'  Off-diag = −edge weight  |  Row sums: {L_wt.sum(axis=1).tolist()}')

print()
print('── 3. Normalized Laplacian D^{{-½}} L D^{{-½}} ───────────────────────────────')
print(pd.DataFrame(L_norm_mat, index=nodes, columns=nodes).round(4).to_string())
print(f'  Diagonal = 1 uniformly: {np.diag(L_norm_mat).round(4).tolist()}')
print(f'  Off-diag (i,j) = -1/√(dᵢ·dⱼ)  |  Eigenvalues bounded [0, 2]')


In [ ]:
# ── Verify all three Laplacian properties hold for every variant ──────────────
# Property 1: rows sum to zero (L measures differences, not absolutes)
# Property 2: smallest eigenvalue = 0 (connected graph → 1 component)
# Property 3: all eigenvalues ≥ 0 (Laplacian is positive semi-definite)
# Property 4: combinatorial only — diagonal = edge count; normalized — diagonal = 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

variants = [
    (L_comb,    'Combinatorial L
diag=degree, off-diag∈{−1,0}'),
    (L_wt,      'Weighted L
diag=Σweights, off-diag=−w_ij'),
    (L_norm_mat,'Normalized L
diag=1, off-diag=−1/√(dᵢdⱼ)'),
]

for ax, (L_var, title) in zip(axes, variants):
    evals_var = np.sort(np.linalg.eigvalsh(L_var))

    # Heatmap of the matrix
    vmax = np.abs(L_var).max()
    im = ax.imshow(L_var, cmap='RdBu', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(nodes, fontsize=8); ax.set_yticklabels(nodes, fontsize=8)
    for i_ in range(n):
        for j_ in range(n):
            ax.text(j_, i_, f'{L_var[i_,j_]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(L_var[i_,j_]) > vmax*0.5 else '#333')
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Verify and annotate
    row_sum_ok  = np.allclose(L_var.sum(axis=1), 0)
    min_eval_ok = abs(evals_var[0]) < 1e-10
    psd_ok      = evals_var[0] >= -1e-10
    ax.set_title(f'{title}
'
                 f'rows→0: {row_sum_ok}  |  λ₁≈0: {min_eval_ok}  |  PSD: {psd_ok}',
                 fontsize=9)

plt.suptitle('Three Laplacian variants: same properties, different entry structure',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

# Eigenvalue comparison table
evals_all = {
    'Combinatorial': np.sort(np.linalg.eigvalsh(L_comb)),
    'Weighted':      np.sort(np.linalg.eigvalsh(L_wt)),
    'Normalized':    np.sort(np.linalg.eigvalsh(L_norm_mat)),
}
eval_df = pd.DataFrame(evals_all, index=[f'λ{i+1}' for i in range(n)]).round(4)
print('Eigenvalue comparison (all λ₁≈0; normalized λ bound [0,2]):')
print(eval_df.to_string())
print()
print('Key differences:')
print('  Combinatorial: eigenvalues scale with degree (no fixed bound)')
print('  Weighted:      eigenvalues scale with weights (can be > combinatorial)')
print('  Normalized:    eigenvalues always in [0, 2] — comparable across networks')


<a id='heat'></a>
---
## Part 3: The Laplacian Intuition: Heat Diffusion on a Graph

> *The Laplacian measures, for every node, how much does this node's value differ from the average of its neighbours' values … In continuous mathematics, the Laplacian appears in the heat equation, describing how temperature spreads through a material. The graph Laplacian is the same idea, discretized.*

We simulate heat diffusion on the five-node network. Start with Ana at temperature 1.0, everyone else at 0. Watch heat spread through the graph edges according to: **dT/dt = −L · T**.

In [ ]:
# ── Discrete heat diffusion: T(t+dt) = T(t) - dt * L * T(t) ──────────────────
dt = 0.05
steps = 60
T = np.zeros(n)
T[idx['Ana']] = 1.0   # Ana starts 'hot'

history = [T.copy()]
for _ in range(steps):
    T = T - dt * (L_mat.astype(float) @ T)
    # Note: no clipping needed — the explicit-Euler scheme is stable when
    #   dt < 2/λ_max.  Here dt=0.05, λ_max≈4.73 → dt·λ_max≈0.24 < 2. ✓
    #   (Stability requires |1 − dt·λ| ≤ 1 for all eigenvalues λ, giving dt ≤ 2/λ_max.)
    #   Clipping would violate conservation of total heat (Σ T = const).
    history.append(T.copy())

history = np.array(history)   # shape: (steps+1, n_nodes)

# ── Plot temperature over time per node ───────────────────────────────────────
colors_nodes = [ACCENT3, NODE_COLOR, ACCENT2, ACCENT1, '#9467bd']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for i, (name, c) in enumerate(zip(nodes, colors_nodes)):
    ax1.plot(history[:, i], label=name, color=c, linewidth=2)
ax1.set_xlabel('Time step'); ax1.set_ylabel('Temperature')
ax1.set_title('Heat diffusion on the graph\n(Ana starts hot, heat spreads via edges)', fontsize=12)
ax1.legend(fontsize=10)
ax1.axhline(1/n, color='black', linestyle='--', linewidth=0.8, label='Equilibrium')
ax1.text(steps*0.85, 1/n+0.01,
         f'equilibrium = 1/n = {1/n:.2f}\n(total heat conserved, spread equally)',
         fontsize=7.5, color='#444')

# Snapshot: temperature on the network at t=15
snap_t = 15
T_snap = history[snap_t]
pos_h  = nx.spring_layout(G, seed=42)
cmap_h = plt.cm.YlOrRd
node_c = [cmap_h(T_snap[idx[v]]) for v in nodes]
nx.draw_networkx(G, pos_h, ax=ax2, node_color=node_c, node_size=700,
                 edge_color=EDGE_COLOR, font_color='black', font_weight='bold', width=2)
for v in nodes:
    x, y = pos_h[v]
    ax2.text(x, y-0.17, f'{T_snap[idx[v]]:.2f}', ha='center', fontsize=9, color='#333')
ax2.set_title(f'Temperature snapshot at step {snap_t}\n(yellow=hot, white=cool)', fontsize=12)
ax2.axis('off')

plt.suptitle('Graph Laplacian as a heat (diffusion) operator', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)
print('Heat diffuses fastest along the densest paths (Bo and Cy heat up quickly).')
print('Dev and Eve — at the periphery — warm up last, mirroring the Fiedler split.')

<a id='fiedler'></a>
---
## Part 4: Fiedler Value, Algebraic Connectivity & Resilience

> *When λ₂ is large, the network is densely knit … When λ₂ equals zero, the network is literally disconnected; more precisely, the number of zero eigenvalues equals the number of disconnected components. When λ₂ has joined λ₁ at zero, analyse each component separately … Simulate node removals and recompute λ₂ for the reduced graph … the people whose removal most dramatically drops algebraic connectivity are the single points of failure.*

**Structural note on the five-node example:** The Fiedler vector splits {Bo, Dev} (negative values) from {Cy, Eve} (positive values), with Ana landing at exactly zero. Ana is the *geometric midpoint* of the two arms in spectral space — she sits on the Fiedler cut boundary, equidistant from both sides. This is a statement about geometric position, not structural vulnerability. The actual articulation points (nodes whose removal disconnects the graph) are **Bo** and **Cy**: removing Bo isolates Dev; removing Cy isolates Eve. Removing Ana leaves all remaining nodes connected. The arms are not symmetric: Dev attaches to Bo, Eve attaches to Cy, so they occupy mirror-image but distinct positions. This is why they appear on *opposite* sides of the cut despite both being leaf nodes.


In [ ]:
G_dense  = nx.watts_strogatz_graph(10, 6, 0.3, seed=1)
G_sparse = nx.watts_strogatz_graph(10, 2, 0.1, seed=2)
G_disc   = nx.Graph()
G_disc.add_edges_from([(0,1),(1,2),(2,3),(3,4),(5,6),(6,7),(7,8),(8,9)])

for label, g in [('Dense (resilient)', G_dense),
                 ('Sparse (fragile)',   G_sparse),
                 ('Disconnected',       G_disc)]:
    evals_g = np.sort(np.linalg.eigvalsh(
        nx.laplacian_matrix(g).toarray().astype(float)))
    n_zeros = np.sum(evals_g < 1e-10)
    lam2    = evals_g[1] if len(evals_g) > 1 else float('nan')
    print(f'{label:25s}  λ₂={lam2:.4f}  '
          f'zero eigenvalues={n_zeros} (= {n_zeros} component{"s" if n_zeros>1 else ""})')

print()
print('Key: #zero eigenvalues = #disconnected components.')
print('When λ₂=0 it has joined λ₁ at zero — the Fiedler value loses meaning.')
print('Always check n_zeros before interpreting λ₂.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (g, title, color) in zip(axes, [
    (G_dense,  'Dense\nλ₂={:.3f} (resilient)', NODE_COLOR),
    (G_sparse, 'Sparse\nλ₂={:.3f} (fragile)',  ACCENT1),
    (G_disc,   'Disconnected\nλ₂={:.3f}',       ACCENT3),
]):
    nx.draw_networkx(g, nx.spring_layout(g, seed=7), ax=ax,
                     node_color=color, edge_color=EDGE_COLOR, node_size=350,
                     font_color='white', font_size=8, font_weight='bold', width=1.5)
    ax.set_title(title.format(fiedler_value(g)), fontsize=11)
    ax.axis('off')
plt.suptitle('λ₂ encodes organisational resilience', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
# ── Node-removal resilience sweep on Karate Club network ──────────────────────
G_kc   = nx.karate_club_graph()
base   = fiedler_value(G_kc)
drops  = []
for nd in G_kc.nodes():
    Gr = G_kc.copy(); Gr.remove_node(nd)
    if nx.is_connected(Gr):
        drops.append((nd, base - fiedler_value(Gr)))
drops.sort(key=lambda x: -x[1])
crit_nodes = [d[0] for d in drops[:8]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.barh([str(d[0]) for d in drops[:8]], [d[1] for d in drops[:8]], color=ACCENT3)
ax1.set_xlabel('Drop in λ₂ after removal'); ax1.invert_yaxis()
ax1.set_title('Most critical nodes\n(Karate Club network)', fontsize=11)

pos_kc = nx.spring_layout(G_kc, seed=5)
nc = [ACCENT3 if nd in crit_nodes else NODE_COLOR for nd in G_kc.nodes()]
nx.draw_networkx(G_kc, pos_kc, ax=ax2, node_color=nc, edge_color=EDGE_COLOR,
                 node_size=280, font_color='white', font_size=7, width=1.1)
ax2.set_title('Red = single points of failure', fontsize=11); ax2.axis('off')
plt.suptitle('Node-removal resilience analysis via Fiedler value', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='sign-degeneracy'></a>
---
## Interlude: Sign Ambiguity and Eigenvalue Degeneracy

### The Fiedler vector sign is arbitrary

The Fiedler vector is defined as a solution to $L v = \lambda_2 v$. If $v$ is a solution, so is $-v$: both satisfy the equation with equal validity. A different run, a different solver, or a different random seed could return the flipped vector, with Bo/Dev positive and Cy/Eve negative.

What is **not** arbitrary: the magnitude of each entry, the Fiedler value $\lambda_2$ itself, and the *relative* signs (nodes on the same side of the cut always share a sign). The notebook applies a sign convention `fv * np.sign(fv[np.argmax(np.abs(fv))])` to make output reproducible, but this is cosmetic. Never interpret the raw sign of a single Fiedler vector entry in isolation.

### What happens when the network is three-fold symmetric?

In the five-node example, Dev hangs off Bo and Eve hangs off Cy, giving the network an asymmetric feel even though the two arms have mirror-image structure. The Fiedler vector finds one clean two-way cut.

Now add **Ana2** as a pendant off Ana, making all three triangle nodes structurally equivalent: each has one leaf attached. The network becomes three-fold symmetric.

> *When the network has three equivalent communities, no single straight cut divides it into two balanced halves better than any other. The Fiedler vector becomes unreliable: $\lambda_2$ is now a **degenerate eigenvalue** shared by two eigenvectors, and any linear combination of them is equally valid. The algorithm picks one essentially at random.*

The eigengap heuristic resolves this: a zero (or near-zero) gap between $\lambda_2$ and $\lambda_3$ signals that you need **two** eigenvectors, not one, and that the network has three natural communities rather than two.

In [ ]:
# ── Three-fold symmetric network: sign ambiguity and eigenvalue degeneracy ────
#
# Original 5-node network: Dev off Bo, Eve off Cy, Ana is a plain hub.
# Extended 6-node network: Ana2 added as pendant off Ana.
# Now every triangle node has exactly one leaf: the network is 3-fold symmetric.

nodes_6 = ['Ana', 'Bo', 'Cy', 'Dev', 'Eve', 'Ana2']
edges_6 = [('Ana','Bo'), ('Ana','Cy'), ('Bo','Cy'),   # triangle core
           ('Bo','Dev'), ('Cy','Eve'), ('Ana','Ana2')]  # one leaf per hub

G6 = nx.Graph()
G6.add_nodes_from(nodes_6)
G6.add_edges_from(edges_6)

idx6 = {name: i for i, name in enumerate(nodes_6)}
n6   = len(nodes_6)
A6   = np.zeros((n6, n6))
for u, v in edges_6:
    A6[idx6[u], idx6[v]] = A6[idx6[v], idx6[u]] = 1
L6 = np.diag(A6.sum(axis=1)) - A6

evals6, evecs6 = eigh(L6)

print('── Original 5-node spectrum ──────────────────────────────────────────────')
print('Eigenvalues:', [f'{v:.4f}' for v in eigenvalues])  # from Part 2
print(f'Gap lambda2 - lambda3: {eigenvalues[2] - eigenvalues[1]:.4f}  <- clear gap, 2-way split valid')

print()
print('── 6-node (3-fold symmetric) spectrum ───────────────────────────────────')
print('Eigenvalues:', [f'{v:.4f}' for v in evals6])
print(f'Gap lambda2 - lambda3: {evals6[2] - evals6[1]:.6f}  <- zero gap, degenerate eigenvalue')
print()
print('lambda2 == lambda3 (exactly):', np.isclose(evals6[1], evals6[2]))
print()
print('Fiedler vector (one arbitrary solution):')
fv6 = evecs6[:, 1]
for name in nodes_6:
    print(f'  {name}: {fv6[idx6[name]]:+.4f}')
print()
print('Note: the entries above are numerically arbitrary within the degenerate')
print('eigenspace. A different solver could return a completely different-looking')
print('vector that is equally correct. The fix is to use BOTH v2 and v3.')


In [ ]:
# ── Using v2 and v3 together correctly recovers the three arms ────────────────

X6 = evecs6[:, 1:3]   # columns: v2, v3 (both span the degenerate eigenspace)
km6 = KMeans(n_clusters=3, n_init=50, random_state=42)
labels6 = km6.fit_predict(X6)

# True communities: {Ana, Ana2}, {Bo, Dev}, {Cy, Eve}
print('Spectral clustering with v2 + v3 (k=3):')
for name in nodes_6:
    print(f'  {name}: cluster {labels6[idx6[name]]}')
print()
print('Expected: Ana+Ana2 together, Bo+Dev together, Cy+Eve together.')

pal6 = [NODE_COLOR, ACCENT1, ACCENT2]
pos6 = nx.spring_layout(G6, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Left: Fiedler vector alone (misleading)
fv6_oriented = fv6 * np.sign(fv6[np.argmax(np.abs(fv6))])
bar_c = [ACCENT1 if fv6_oriented[idx6[n]] > 0.01 else
         (ACCENT3 if fv6_oriented[idx6[n]] < -0.01 else '#888888')
         for n in nodes_6]
nx.draw_networkx(G6, pos6, ax=axes[0],
                 node_color=bar_c, edge_color=EDGE_COLOR,
                 font_color='white', font_weight='bold', node_size=700, width=2)
axes[0].set_title('Fiedler vector alone (v2 only)\nArbitrary: depends on solver', fontsize=11)
axes[0].axis('off')

# Middle: v2 vs v3 scatter
axes[1].scatter(X6[:, 0], X6[:, 1],
                c=[pal6[labels6[idx6[n]]] for n in nodes_6],
                s=200, zorder=3)
for name in nodes_6:
    axes[1].annotate(name,
                     (X6[idx6[name], 0] + 0.01, X6[idx6[name], 1] + 0.01),
                     fontsize=9)
axes[1].set_xlabel('v2 (Fiedler)')
axes[1].set_ylabel('v3')
axes[1].set_title('Spectral embedding (v2, v3)\nThree arms separate cleanly', fontsize=11)
axes[1].axhline(0, color='#ddd', linewidth=0.8)
axes[1].axvline(0, color='#ddd', linewidth=0.8)

# Right: correctly recovered communities
nx.draw_networkx(G6, pos6, ax=axes[2],
                 node_color=[pal6[labels6[idx6[n]]] for n in nodes_6],
                 edge_color=EDGE_COLOR,
                 font_color='white', font_weight='bold', node_size=700, width=2)
axes[2].set_title('Communities from v2 + v3\n{Ana,Ana2} {Bo,Dev} {Cy,Eve}', fontsize=11)
axes[2].axis('off')

plt.suptitle('Degenerate eigenvalue: v2 alone is unreliable; v2+v3 together recover all three arms',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
plt.close(fig)

print()
print('Key takeaways:')
print('  1. When lambda2 = lambda3, the Fiedler vector is not uniquely defined.')
print('  2. The eigengap heuristic (zero gap) correctly signals k=3 communities.')
print('  3. Using both eigenvectors spanning the degenerate space recovers all three')
print('     communities perfectly, just as standard spectral clustering would.')
print('  4. Always check the eigengap before interpreting a Fiedler vector as a')
print('     definitive two-way partition.')


<a id='spectral-clustering'></a>
---
## Part 5: Spectral Clustering: Finding Informal Communities

> *The third, fourth, and fifth eigenvectors give finer partitions … spectral clustering … discovers informal teams, communities of practice, or silos that do not appear on any org chart. The Fiedler value tells you how severe the silos are.*

We generate a 36-node stochastic block model with three hidden communities and recover them using Laplacian eigenvectors.

In [ ]:
sizes    = [12, 14, 10]
p_matrix = [[0.55, 0.04, 0.04],
            [0.04, 0.55, 0.04],
            [0.04, 0.04, 0.55]]
G_sbm = nx.stochastic_block_model(sizes, p_matrix, seed=42)
true_labels = [i for i, s in enumerate(sizes) for _ in range(s)]

L_sbm = nx.laplacian_matrix(G_sbm).toarray().astype(float)
evals_sbm, evecs_sbm = eigh(L_sbm)

# k-means on eigenvectors 2, 3, 4
km = KMeans(n_clusters=3, n_init=20, random_state=0)
pred = km.fit_predict(evecs_sbm[:, 1:4])

# Find the label permutation that best aligns predicted clusters to true communities.
# p[i] = which predicted cluster maps to true community i.
# Predicted label l → true label inverse_p[l], where inverse_p[p[i]] = i.
def perm_accuracy(pred, true, p):
    inv = {p[i]: i for i in range(3)}   # predicted → true
    return np.mean([inv[l] for l in pred] == np.array(true))

best_acc = max(perm_accuracy(pred, true_labels, list(p))
               for p in permutations([0,1,2]))
print(f'Spectral clustering accuracy: {best_acc:.1%}  (on {sum(sizes)}-node network)')

In [ ]:
pal = [NODE_COLOR, ACCENT1, ACCENT2]
pos_sbm = nx.spring_layout(G_sbm, seed=20)

# Remap predicted labels to best-matching true labels for visualisation
best_perm = max(permutations([0,1,2]),
                key=lambda p: perm_accuracy(pred, true_labels, list(p)))
inv_best = {best_perm[i]: i for i in range(3)}   # predicted → true
recovered = [inv_best[l] for l in pred]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (labels, title) in zip(axes, [
    (true_labels, 'True communities\n(ground truth)'),
    (true_labels, 'Spectral embedding\n(click to see scatter)'),
    (recovered,   f'Spectral clustering\n(accuracy {best_acc:.0%})'),
]):
    if 'embedding' in title:
        axes[1].scatter(evecs_sbm[:,1], evecs_sbm[:,2],
                        c=[pal[l] for l in true_labels], s=50, alpha=0.8)
        axes[1].set_xlabel('2nd eigenvector (Fiedler)')
        axes[1].set_ylabel('3rd eigenvector')
        axes[1].set_title(title, fontsize=11)
    else:
        nx.draw_networkx(G_sbm, pos_sbm, ax=ax,
                         node_color=[pal[l] for l in labels],
                         edge_color=EDGE_COLOR, node_size=200,
                         with_labels=False, width=0.7)
        ax.set_title(title, fontsize=11); ax.axis('off')

plt.suptitle('Spectral clustering recovers hidden organisational communities', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='spectral-drawing'></a>
---
## Part 6: Spectral Graph Drawing: Eigenvectors as Geometry

> *Use v₂ and v₃ as x/y coordinates to draw the network … the mathematically optimal 2D layout in a precise sense. By the Rayleigh-Ritz characterisation (subject to the constraint x ⊥ 1), v₂ minimises the Laplacian quadratic form … Together they produce a layout where nodes connected by edges are pulled as close together as possible globally.*
>
> *When λ₂ is large, no drawing algorithm will produce a clean picture, and that is itself a finding. A high Fiedler value means the organisation is well-integrated without strong cluster structure.*

We demonstrate three things:
1. **Hall's method** on the essay's five-person network (eigenvectors as coordinates)
2. **Higher dimensions**: v₂, v₃, v₄ for a 3D embedding of a 3-community graph
3. **The λ₂ ↔ drawability theorem**: structured (low-λ₂) vs random expander (high-λ₂)


In [ ]:
# ── 1. Hall's spectral layout on the five-person essay network ─────────────────
# v₁ = constant (trivial), v₂ = Fiedler, v₃ = next structural dimension.
# G, L_mat, eigenvectors, nodes are defined in Part 2.

evals_draw, evecs_draw = eigh(L_mat)
v2 = evecs_draw[:, 1]   # x-coordinates (Fiedler vector)
v3 = evecs_draw[:, 2]   # y-coordinates

pos_spectral = {name: (v2[i], v3[i]) for i, name in enumerate(nodes)}
pos_spring   = nx.spring_layout(G, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Standard spring layout
nx.draw_networkx(G, pos_spring, ax=axes[0],
                 node_color=NODE_COLOR, edge_color=EDGE_COLOR,
                 font_color='white', font_weight='bold', node_size=700, width=2)
axes[0].set_title('Spring layout\n(force-directed, arbitrary geometry)', fontsize=11)
axes[0].axis('off')

# Hall spectral layout
nx.draw_networkx(G, pos_spectral, ax=axes[1],
                 node_color=ACCENT1, edge_color=EDGE_COLOR,
                 font_color='white', font_weight='bold', node_size=700, width=2)
axes[1].set_title('Spectral layout (Hall)\n(v₂ = x-axis, v₃ = y-axis)', fontsize=11)
axes[1].axis('off')
for name, (x, y) in pos_spectral.items():
    axes[1].text(x, y - 0.08, f'v₂={v2[nodes.index(name)]:+.2f}',
                 ha='center', fontsize=7, color='#333')

# Eigenvector values as bar chart
x_idx = np.arange(len(nodes))
axes[2].bar(x_idx - 0.2, v2, 0.35, label='v₂  (x-coord)', color=ACCENT1)
axes[2].bar(x_idx + 0.2, v3, 0.35, label='v₃  (y-coord)', color=ACCENT2)
axes[2].set_xticks(x_idx); axes[2].set_xticklabels(nodes)
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_ylabel('Eigenvector value')
axes[2].set_title('Eigenvector coordinates\n(these ARE the node positions)', fontsize=11)
axes[2].legend(fontsize=9)

plt.suptitle("Hall's spectral graph drawing: eigenvectors as spatial coordinates",
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

print("The Fiedler vector (v₂) separates Dev/Eve from the core — same split as community")
print("detection. Layout and clustering are two faces of the same spectral geometry.")


In [ ]:
# ── 2. Higher-dimensional embedding: 3 communities → 3 eigenvectors ───────────
# v₂, v₃, v₄ as x, y, z gives a 3D layout. k communities → k eigenvectors.

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

sizes_3d    = [15, 15, 15]
p_matrix_3d = [[0.6, 0.03, 0.03],
               [0.03, 0.6, 0.03],
               [0.03, 0.03, 0.6]]
G_3d     = nx.stochastic_block_model(sizes_3d, p_matrix_3d, seed=7)
true_3d  = [i for i, s in enumerate(sizes_3d) for _ in range(s)]
L_3d     = nx.laplacian_matrix(G_3d).toarray().astype(float)
evals_3d, evecs_3d = eigh(L_3d)
xyz      = evecs_3d[:, 1:4]   # columns: v₂, v₃, v₄

pal_3d   = [NODE_COLOR, ACCENT1, ACCENT2]
colors_3d = [pal_3d[l] for l in true_3d]
pos2d_3d  = {i: (evecs_3d[i,1], evecs_3d[i,2]) for i in range(sum(sizes_3d))}

fig = plt.figure(figsize=(14, 5))

# 2D spectral layout
ax2d = fig.add_subplot(131)
nx.draw_networkx(G_3d, pos2d_3d, ax=ax2d,
                 node_color=colors_3d, edge_color=EDGE_COLOR,
                 node_size=80, with_labels=False, width=0.5)
ax2d.set_title(f'2D spectral layout (v₂, v₃)\nλ₂={evals_3d[1]:.3f}', fontsize=11)
ax2d.axis('off')

# 3D spectral embedding
ax3d = fig.add_subplot(132, projection='3d')
for i, (x, y, z) in enumerate(xyz):
    ax3d.scatter(x, y, z, color=colors_3d[i], s=40, alpha=0.85)
for u, v_node in G_3d.edges():
    ax3d.plot([xyz[u,0], xyz[v_node,0]],
              [xyz[u,1], xyz[v_node,1]],
              [xyz[u,2], xyz[v_node,2]],
              color=EDGE_COLOR, alpha=0.3, linewidth=0.6)
ax3d.set_title('3D spectral embedding\n(v₂, v₃, v₄)', fontsize=11)
ax3d.set_xlabel('v₂', fontsize=8); ax3d.set_ylabel('v₃', fontsize=8)
ax3d.set_zlabel('v₄', fontsize=8); ax3d.tick_params(labelsize=6)

# Spectrum showing the gap
ax_ev = fig.add_subplot(133)
highlight = [ACCENT3 if i < 3 else NODE_COLOR for i in range(8)]
ax_ev.bar(range(1, 9), evals_3d[1:9], color=highlight)
ax_ev.set_xticks(range(1, 9))
ax_ev.set_xticklabels([f'λ{i+1}' for i in range(8)])
ax_ev.set_ylabel('Eigenvalue')
ax_ev.set_title('Spectrum: gap after λ₄\n(signals k=3 communities)', fontsize=11)
ax_ev.axhline(0, color='black', linewidth=0.5, linestyle='--')

plt.suptitle('Higher-dimensional spectral embedding: k communities → k eigenvectors',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

print(f"Eigenvalue gap: λ₄={evals_3d[3]:.3f} → λ₅={evals_3d[4]:.3f}")
print("k communities leave k-1 near-zero eigenvalues, then a gap before λ_{k+1}.")
print("The number of meaningful embedding dimensions equals the number of communities.")


In [ ]:
# ── 3a. The λ₂ ↔ drawability theorem: data collection ──────────────────────
# Structured graph (low λ₂, should draw cleanly) vs random 4-regular expander
# (high λ₂, provably un-drawable by the informal Spielman theorem).

def spectral_pos_2d(G):
    L = nx.laplacian_matrix(G).toarray().astype(float)
    _, ev = eigh(L)
    return {i: (ev[i,1], ev[i,2]) for i in range(G.number_of_nodes())}

def mean_edge_length(G, pos):
    lengths = [np.linalg.norm(np.array(pos[u]) - np.array(pos[v]))
               for u, v in G.edges()]
    return float(np.mean(lengths)) if lengths else 0.0

# Structured: 3-community SBM, 36 nodes
G_struct  = nx.stochastic_block_model([12,12,12],
                                      [[0.6,0.03,0.03],
                                       [0.03,0.6,0.03],
                                       [0.03,0.03,0.6]], seed=1)
true_struct = [i for i, s in enumerate([12,12,12]) for _ in range(s)]

# Random: 4-regular expander, same size
G_rand    = nx.random_regular_graph(4, 36, seed=42)

lam2_s = fiedler_value(G_struct)
lam2_r = fiedler_value(G_rand)
pos_s  = spectral_pos_2d(G_struct)
pos_r  = spectral_pos_2d(G_rand)
mel_s  = mean_edge_length(G_struct, pos_s)
mel_r  = mean_edge_length(G_rand,   pos_r)

n_nodes   = G_rand.number_of_nodes()
threshold = 10.0 / np.sqrt(n_nodes)

print(f"Structured graph  λ₂={lam2_s:.4f}  mean edge length={mel_s:.4f}")
print(f"Random 4-regular  λ₂={lam2_r:.4f}  mean edge length={mel_r:.4f}")
print()
print(f"Informal threshold for n={n_nodes}: λ₂ > {threshold:.3f} → no clean layout")
print(f"Random graph λ₂={lam2_r:.3f}: {'ABOVE — no clean drawing possible' if lam2_r > threshold else 'below threshold'}")


In [ ]:
# ── 3b. Visual comparison and λ₂ sweep ──────────────────────────────────────

pal_s2 = [NODE_COLOR, ACCENT1, ACCENT2]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Spectral — structured (should look clean)
nx.draw_networkx(G_struct, pos_s, ax=axes[0],
                 node_color=[pal_s2[l] for l in true_struct],
                 edge_color=EDGE_COLOR, node_size=120, with_labels=False, width=0.8)
axes[0].set_title(f'Structured: spectral layout\nλ₂={lam2_s:.3f} (low → drawable)',
                  fontsize=10); axes[0].axis('off')

# Spectral — random (should look tangled)
nx.draw_networkx(G_rand, pos_r, ax=axes[1],
                 node_color=ACCENT3, edge_color=EDGE_COLOR,
                 node_size=120, with_labels=False, width=0.8)
axes[1].set_title(f'Random 4-regular: spectral layout\nλ₂={lam2_r:.3f} (high → undrawable)',
                  fontsize=10); axes[1].axis('off')

# Spring — random (proves the problem is the graph, not the algorithm)
pos_r_spring = nx.spring_layout(G_rand, seed=1)
nx.draw_networkx(G_rand, pos_r_spring, ax=axes[2],
                 node_color=ACCENT3, edge_color=EDGE_COLOR,
                 node_size=120, with_labels=False, width=0.8)
axes[2].set_title('Random 4-regular: spring layout\n(still tangled — the graph, not the algorithm)',
                  fontsize=10); axes[2].axis('off')

# Sweep: rising λ₂ → longer mean edge lengths → worse drawings
lambdas_sw, edgelens_sw = [], []
for pw_sw in np.linspace(0.05, 0.65, 18):
    pc_sw = max(0.01, 0.7 - pw_sw)
    try:
        Gs_sw = nx.stochastic_block_model([12,12,12],
                                          [[pw_sw,pc_sw,pc_sw],
                                           [pc_sw,pw_sw,pc_sw],
                                           [pc_sw,pc_sw,pw_sw]], seed=3)
        if nx.is_connected(Gs_sw):
            lv = fiedler_value(Gs_sw)
            ps_sw = spectral_pos_2d(Gs_sw)
            lambdas_sw.append(lv)
            edgelens_sw.append(mean_edge_length(Gs_sw, ps_sw))
    except Exception:
        pass

axes[3].scatter(lambdas_sw, edgelens_sw, color=NODE_COLOR, s=50, alpha=0.8)
axes[3].axvline(lam2_s, color=ACCENT2, linestyle='--', linewidth=1.5,
                label=f'Structured λ₂={lam2_s:.2f}')
axes[3].axvline(lam2_r, color=ACCENT3, linestyle='--', linewidth=1.5,
                label=f'Random λ₂={lam2_r:.2f}')
axes[3].set_xlabel('Fiedler value λ₂')
axes[3].set_ylabel('Mean edge length in spectral layout')
axes[3].set_title('Higher λ₂ → longer edges → worse picture\n(λ₂ predicts drawing quality)',
                  fontsize=10)
axes[3].legend(fontsize=8)

plt.suptitle('λ₂ ↔ drawability: Fiedler value determines whether a clean layout is possible',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

print()
print("ONA practical rule:")
print(f"  Compute λ₂ before presenting a network map.")
print(f"  For n={n_nodes}: λ₂ < {threshold:.2f} → layout will be interpretable.")
print("  λ₂ above that → high connectivity IS the finding; no layout can show cluster structure.")
print("  A tangled network diagram is not a visualisation failure — it is a structural result.")


<a id='directed'></a>
---
## Part 8: Directed Networks: In-Centrality and Out-Centrality

> *Eigenvector centrality splits into two measures: in-centrality (HITS authority, that is, the leading eigenvector of AᵀA, which measures who receives attention from high-status nodes) and out-centrality (HITS hub, that is, the leading eigenvector of AAᵀ, which measures who sends attention toward high-status nodes). PageRank exists precisely because Google needed a principled directed-graph centrality.*

We build a directed reporting/influence network and show that in-centrality and out-centrality tell fundamentally different stories about the same people.

**Note on directed Laplacians:** The directed Laplacian L = D_out − A is not symmetric. Its eigenvalues can be complex numbers, which breaks the real spectral decomposition framework used throughout the rest of this notebook. This is why directed graph analysis relies on HITS and PageRank rather than Laplacian eigenvectors.

In [ ]:
# ── Directed org influence network ────────────────────────────────────────────
# Edges represent: 'i influences / reports to j'
D_nodes = ['CEO','VP-Eng','VP-Sales','Lead-A','Lead-B','Lead-C',
           'Dev-1','Dev-2','AE-1','AE-2','AE-3']
D_edges = [
    ('VP-Eng','CEO'), ('VP-Sales','CEO'),
    ('Lead-A','VP-Eng'), ('Lead-B','VP-Eng'), ('Lead-C','VP-Sales'),
    ('Dev-1','Lead-A'), ('Dev-2','Lead-A'), ('Dev-2','Lead-B'),
    ('AE-1','Lead-C'), ('AE-2','Lead-C'), ('AE-3','Lead-C'),
    # Informal cross-flows
    ('Dev-2','Lead-C'), ('AE-1','VP-Eng'),
]
DG = nx.DiGraph()
DG.add_nodes_from(D_nodes)
DG.add_edges_from(D_edges)

# In-centrality: who receives from high-status nodes (HITS authority)
hits_hub, hits_auth = nx.hits(DG, max_iter=500)
# PageRank: directed random-walk influence
pr = nx.pagerank(DG, alpha=0.85)
# Out-degree (who pushes information out)
out_deg = dict(DG.out_degree())
in_deg  = dict(DG.in_degree())

dir_df = pd.DataFrame({
    'In-degree':  in_deg,
    'Out-degree': out_deg,
    'HITS Auth (in-centrality)':  hits_auth,
    'HITS Hub  (out-centrality)': hits_hub,
    'PageRank':   pr,
}).round(4).sort_values('PageRank', ascending=False)
print(dir_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pos_d = nx.spring_layout(DG, seed=12)

for ax, (vals, title, cmap) in zip(axes, [
    (hits_auth, 'HITS Authority (in-centrality)\nWho receives from important nodes', 'Blues'),
    (hits_hub,  'HITS Hub (out-centrality)\nWho sends to important nodes',           'Oranges'),
    (pr,        'PageRank\nDirected random-walk influence',                           'Greens'),
]):
    v_arr  = np.array([vals[nd] for nd in DG.nodes()])
    v_norm = (v_arr - v_arr.min()) / (v_arr.max() - v_arr.min() + 1e-9)
    cm     = matplotlib.colormaps.get_cmap(cmap)
    nc     = [cm(0.3 + 0.7*v) for v in v_norm]
    sz     = [150 + 1200*v for v in v_norm]
    nx.draw_networkx(DG, pos_d, ax=ax, node_color=nc, node_size=sz,
                     edge_color=EDGE_COLOR, font_size=7, font_color='white',
                     arrows=True, arrowsize=12, width=1.2, with_labels=True)
    ax.set_title(title, fontsize=10); ax.axis('off')
    top = max(vals, key=vals.get)
    ax.text(0.5, -0.04, f'Top: {top}', transform=ax.transAxes,
            ha='center', fontsize=9, color='#444')

plt.suptitle('Directed network: in-centrality vs out-centrality tell different stories',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)
print('Arrow direction: i → j means i reports to / influences j.')
print('Note: Dev-2 and AE-1 score high on hub centrality due to their cross-layer informal ties.')

<a id='centrality'></a>
---
## Part 9: Centrality on Undirected Networks

> *Degree centrality: raw number of connections … Eigenvector centrality: your score is proportional to the sum of your neighbours' scores … Betweenness: what fraction of shortest paths between all pairs pass through you? … PageRank adds a decay parameter.*

Four measures on a 20-node Barabási–Albert network, plus a correlation heatmap showing when they agree and diverge.

In [ ]:
G_org = nx.barabasi_albert_graph(20, 2, seed=7)
deg_c   = nx.degree_centrality(G_org)
eig_c   = nx.eigenvector_centrality(G_org, max_iter=500)
btw_c   = nx.betweenness_centrality(G_org)
pr_c    = nx.pagerank(G_org, alpha=0.85)

c_df = pd.DataFrame({'Degree': deg_c, 'Eigenvector': eig_c,
                     'Betweenness': btw_c, 'PageRank': pr_c
                    }).sort_values('Eigenvector', ascending=False)
print('Top 8 nodes by eigenvector centrality:')
print(c_df.head(8).round(4).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
pos_org = nx.spring_layout(G_org, seed=3)

for ax, (measure, title, cmap) in zip(axes.flat, [
    (deg_c, 'Degree centrality\n(raw connections)',           'Blues'),
    (eig_c, 'Eigenvector centrality\n(well-connected neighbours)', 'Oranges'),
    (btw_c, 'Betweenness centrality\n(bottleneck / broker)',   'Reds'),
    (pr_c,  'PageRank\n(directed-flow influence)',             'Greens'),
]):
    vals  = np.array([measure[nd] for nd in G_org.nodes()])
    vnorm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    cm    = matplotlib.colormaps.get_cmap(cmap)
    nx.draw_networkx(G_org, pos_org, ax=ax,
                     node_color=[cm(0.3+0.7*v) for v in vnorm],
                     node_size=[200+1400*v for v in vnorm],
                     edge_color=EDGE_COLOR, font_size=8,
                     font_color='white', font_weight='bold', width=1)
    ax.set_title(title, fontsize=11); ax.axis('off')
    top = max(measure, key=measure.get)
    ax.text(0.5, -0.03, f'Top: node {top}  ({measure[top]:.3f})',
            transform=ax.transAxes, ha='center', fontsize=9, color='#444')

plt.suptitle('Four centrality measures: darker/larger = more central', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
corr = c_df.corr()
im = ax.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
labs = ['Degree', 'Eigenvec.', 'Betweenness', 'PageRank']
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(labs, rotation=30, ha='right'); ax.set_yticklabels(labs)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Centrality measure correlations', fontsize=11)
plt.tight_layout(); plt.show(); plt.close(fig)
print('Nodes that rank high on betweenness but low on eigenvector are the brokers —')
print('they sit between cliques rather than inside them.')

<a id='powers'></a>
---
## Part 10: Powers of the Adjacency Matrix: Walks, Triangles & Structural Holes

> *The entry in row i, column j of A² gives the number of walks of length exactly 2 … A³ diagonal entries each equal twice the number of triangles … High triangle counts mean dense clique membership. Low triangle counts with high degree identify brokers: Burt's structural holes.*

In [ ]:
# Matrix powers: A² counts 2-step walks, A³ counts 3-step walks.
# Using A as float (as constructed) — division by 2 uses / not // to avoid
# any floating-point rounding issues with the diagonal.
A2 = A @ A
A3 = A @ A @ A

print('A² — walks of length 2 (shared contacts):')
print(pd.DataFrame(A2, index=nodes, columns=nodes).to_string())
# Each triangle contributes 2 to the diagonal of A³ for each of its vertices
# (one for each direction around the triangle), so we divide by 2.
print('\nTriangle counts per node  (A³ diagonal / 2):')
for name, tri in zip(nodes, np.diag(A3) / 2):
    nx_tri = nx.triangles(G, name)
    status = '✓' if int(tri) == nx_tri else '✗ MISMATCH'
    print(f'  {name}: matrix={int(tri)}  networkx={nx_tri}  {status}')
print('\nConclusion: Ana, Bo, Cy form the only triangle.  Dev and Eve are bridge endpoints.')

In [ ]:
constraint = nx.constraint(G)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

inv_c  = {nd: 1.0 / (constraint[nd] + 0.01) for nd in G.nodes()}
max_ic = max(inv_c.values())
nx.draw_networkx(G, pos, ax=ax1,
                 node_size=[200+2000*inv_c[nd]/max_ic for nd in G.nodes()],
                 node_color=NODE_COLOR, edge_color=EDGE_COLOR,
                 font_color='white', font_weight='bold')
ax1.set_title('Node size ∝ structural holes\n(larger = more broker/unconstrained)', fontsize=11)
ax1.axis('off')

ax2.barh(nodes, [constraint[nd] for nd in nodes],
         color=[ACCENT3 if constraint[nd] < 0.5 else NODE_COLOR for nd in nodes])
ax2.axvline(0.5, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel("Burt's constraint (lower = more structural holes)")
ax2.set_title('Constraint index per node', fontsize=11); ax2.invert_yaxis()

plt.suptitle("Structural holes: who bridges disconnected worlds?", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='random-walk'></a>
---
## Part 11: Normalised Laplacian & Random Walk Mixing

> *The spectral gap controls how fast the random walk mixes. A large spectral gap means the walk quickly reaches every part of the network: the organisation diffuses information rapidly. A small spectral gap means the walk gets stuck in local neighbourhoods.*

In [ ]:
def rw_coverage(G, steps=80, n_walkers=400, seed=0):
    """
    Fraction of distinct nodes reached by random walkers at each step.

    Vectorised implementation: O(steps × n_walkers) instead of O(steps²).
    All walkers advance together one step per iteration — no restarts.
    Old implementation restarted every walker from scratch for each step
    count s, costing O(steps²) Python iterations (3,240 passes for steps=80).
    """
    rng   = np.random.default_rng(seed)
    nlist = np.array(list(G.nodes()))
    n     = len(nlist)
    nidx  = {v: i for i, v in enumerate(nlist)}

    # Build padded adjacency array: adj_mat[i, :deg[i]] = neighbour indices
    max_deg = max(d for _, d in G.degree())
    adj_mat = np.full((n, max_deg), -1, dtype=np.int32)
    deg_arr = np.zeros(n, dtype=np.int32)
    for v in nlist:
        nbrs = list(G.neighbors(v))
        i    = nidx[v]
        deg_arr[i] = len(nbrs)
        adj_mat[i, :len(nbrs)] = [nidx[u] for u in nbrs]

    # Start walkers at random positions, advance all simultaneously
    current = rng.integers(0, n, size=n_walkers)
    cov = []
    for _ in range(steps):
        rand_col = (rng.random(n_walkers) * deg_arr[current]).astype(np.int32)
        current  = adj_mat[current, rand_col]
        cov.append(len(np.unique(current)) / n)
    return cov

G_fast = nx.watts_strogatz_graph(30, 8, 0.5, seed=1)
G_slow = nx.watts_strogatz_graph(30, 2, 0.05, seed=2)
sg_fast = normalized_laplacian_spectrum(G_fast)[1]
sg_slow = normalized_laplacian_spectrum(G_slow)[1]
print(f'Fast-mixing spectral gap: {sg_fast:.4f}')
print(f'Slow-mixing spectral gap: {sg_slow:.4f}')

cov_fast = rw_coverage(G_fast)
cov_slow = rw_coverage(G_slow)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 81), cov_fast, color=ACCENT2, linewidth=2.5,
        label=f'Large gap ({sg_fast:.3f}) — fast mixing')
ax.plot(range(1, 81), cov_slow, color=ACCENT3, linewidth=2.5,
        label=f'Small gap ({sg_slow:.3f}) — slow mixing')
ax.set_xlabel('Random walk steps'); ax.set_ylabel('Fraction of nodes reached')
ax.set_title('Spectral gap predicts information diffusion speed', fontsize=12)
ax.legend(fontsize=10); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='cospectral'></a>
---
## Part 12: Cospectral Graphs: Limits of the Spectral Fingerprint

> *Two networks with the same characteristic polynomial are called cospectral; they share all spectral properties but are not necessarily identical in structure. Spectral signatures are best used for tracking change over time within the same network rather than as a definitive structural identifier across different ones.*

We display a verified pair of adjacency-cospectral graphs on 6 nodes and confirm they share the same eigenvalues despite being non-isomorphic.

In [ ]:
# ── A verified adjacency-cospectral pair (6 nodes each) ──────────────────────
#
# Both graphs have adjacency spectrum {-√3, -1, 0, 0, 1, √3}
# but are non-isomorphic — different degree sequences, different structure.
# This pair is exhaustively verified from the graph atlas.
#
# G1: a branched path (degrees: 0, 1, 1, 2, 2, 2)
#     One isolated node (5). Edges form a path 0-2-1 plus chain 1-3-5.
G_cs1 = nx.Graph()
G_cs1.add_nodes_from(range(6))
G_cs1.add_edges_from([(0,2),(1,2),(1,3),(3,5)])   # node 4 is isolated

# G2: a star K_{1,3} plus one separate edge  (degrees: 1, 1, 1, 1, 1, 3)
#     Centre node 0 connects to leaves 3,4,5; separate edge 1-2.
G_cs2 = nx.Graph()
G_cs2.add_nodes_from(range(6))
G_cs2.add_edges_from([(0,3),(0,4),(0,5),(1,2)])

def adj_spectrum(G):
    n_nodes = G.number_of_nodes()
    A_mat = np.zeros((n_nodes, n_nodes))
    node_list = sorted(G.nodes())
    nidx = {v: i for i, v in enumerate(node_list)}
    for u, v in G.edges():
        A_mat[nidx[u], nidx[v]] = 1
        A_mat[nidx[v], nidx[u]] = 1
    return np.round(np.sort(np.linalg.eigvalsh(A_mat)), 6)

sp1 = adj_spectrum(G_cs1)
sp2 = adj_spectrum(G_cs2)
print('Graph 1 adjacency spectrum:', sp1)
print('Graph 2 adjacency spectrum:', sp2)
print(f'Spectra identical: {np.allclose(sp1, sp2)}')
print(f'Graphs isomorphic: {nx.is_isomorphic(G_cs1, G_cs2)}')

# Important nuance: these graphs are cospectral w.r.t. the ADJACENCY matrix A.
# Their LAPLACIAN spectra are different — check:
from scipy.sparse.csgraph import laplacian as csg_laplacian
def lap_spectrum(G):
    A_mat = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
    return np.round(np.sort(np.linalg.eigvalsh(csg_laplacian(A_mat))), 6)
lsp1 = lap_spectrum(G_cs1)
lsp2 = lap_spectrum(G_cs2)
print('\nLaplacian spectrum G1:', lsp1)
print('Laplacian spectrum G2:', lsp2)
print(f'Laplacian spectra identical: {np.allclose(lsp1, lsp2)}')
print('\nKey insight: A-cospectral ≠ L-cospectral. Different matrices,\n'
      'different fingerprints. Always specify WHICH matrix when discussing cospectrality.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, g, title in [
    (axes[0], G_cs1, 'Graph 1: branched path + isolate\n(degrees: 0,1,1,2,2,2)'),
    (axes[1], G_cs2, 'Graph 2: K₁,₃ + K₂\n(star + separate edge; degrees: 1,1,1,1,1,3)'),
]:
    nx.draw_networkx(g, nx.spring_layout(g, seed=9), ax=ax,
                     node_color=NODE_COLOR, edge_color=EDGE_COLOR,
                     node_size=450, font_color='white', font_weight='bold', width=2)
    ax.set_title(title, fontsize=11); ax.axis('off')

# Spectrum comparison
x = np.arange(len(sp1))
axes[2].bar(x - 0.2, sp1, 0.35, label='Graph 1', color=NODE_COLOR)
axes[2].bar(x + 0.2, sp2, 0.35, label='Graph 2', color=ACCENT1)
axes[2].set_title('Identical spectra\n(same bars, different structures!)', fontsize=11)
axes[2].set_xlabel('Eigenvalue index'); axes[2].set_ylabel('Eigenvalue')
axes[2].legend(fontsize=9)

plt.suptitle('Cospectral graphs: same spectrum, different structure — spectral fingerprints have limits',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)
print('Implication for ONA: use spectral tracking within one network over time,')
print('not to declare two different organisations "structurally equivalent".')

<a id='bipartite'></a>
---
## Part 13: Bipartite Networks: People Connecting Through Things

> *A large and analytically rich class of organizational data … you observe people connected to things such as documents, projects, Slack threads … P = B·Bᵀ … The standard correction is to weight each co-membership by the inverse size of the shared artifact … Jaccard similarity [further normalises] for people with very different activity levels.*

We build B (people × projects), compute raw, inverse-size-weighted, and Jaccard-normalised projections, then apply SVD.

In [ ]:
people_b   = ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hana']
projects_b = ['Proj-A','Proj-B','Proj-C','Proj-D','Proj-E','Proj-F']

B = np.array([
    [1,1,0,0,0,0],  # Alice
    [1,1,1,0,0,0],  # Bob   — cross-team connector
    [0,1,1,0,0,0],  # Carol
    [0,0,1,1,0,0],  # Dave
    [0,0,0,1,1,0],  # Eve
    [0,0,0,1,1,1],  # Frank — cross-team connector
    [0,0,0,0,1,1],  # Grace
    [0,0,1,0,0,1],  # Hana  — rare bridge A↔B cluster
], dtype=float)
m_b, n_b = B.shape

# Raw projection
P_raw = B @ B.T;  np.fill_diagonal(P_raw, 0)

# Inverse-size weighted
proj_sizes = B.sum(axis=0)
P_wt = sum(np.outer(B[:,k], B[:,k]) / proj_sizes[k] for k in range(n_b))
np.fill_diagonal(P_wt, 0)

# Jaccard similarity: |A∩B| / |A∪B| = co-memberships / (breadth_i + breadth_j − co-memberships)
# We use P_raw[i,j] for co-membership count (safe: diagonal was zeroed, off-diag correct).
# Using P_raw (not P_wt) keeps the numerator as a raw count matching the denominator's units.
breadth = B.sum(axis=1)   # number of projects per person
P_jac   = np.zeros_like(P_raw)
for i in range(m_b):
    for j in range(m_b):
        if i == j: continue   # leave diagonal as zero
        denom = breadth[i] + breadth[j] - P_raw[i,j]  # = |A ∪ B|
        P_jac[i,j] = P_raw[i,j] / denom if denom > 0 else 0
np.fill_diagonal(P_jac, 0)

print('Project sizes (s_k):', dict(zip(projects_b, proj_sizes.astype(int))))
print('\nPerson activity breadth (# projects):', dict(zip(people_b, breadth.astype(int))))
print('\nJaccard matrix (normalised for activity level):')
print(pd.DataFrame(P_jac.round(3), index=people_b, columns=people_b).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, mat, title in [
    (axes[0], P_raw, 'Raw projection P=B·Bᵀ\n(shared project count)'),
    (axes[1], P_wt,  'Inverse-size weighted\n(discounts large projects)'),
    (axes[2], P_jac, 'Jaccard normalised\n(adjusts for activity breadth)'),
]:
    im = ax.imshow(mat, cmap='Blues')
    ax.set_xticks(range(m_b)); ax.set_yticks(range(m_b))
    ax.set_xticklabels(people_b, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(people_b, fontsize=7)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Three normalisation strategies for bipartite projection', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

In [ ]:
# ── SVD: place people AND projects in the same latent space ───────────────────
U, s_vals, Vt = svd(B, full_matrices=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.bar(range(1, len(s_vals)+1), s_vals, color=NODE_COLOR)
ax1.set_xlabel('Singular value index'); ax1.set_ylabel('Singular value')
ax1.set_title('SVD singular values\n(captures shared structure)', fontsize=11)

ax2.scatter(U[:,0], U[:,1], color=NODE_COLOR, s=100, zorder=3, label='People')
for i, name in enumerate(people_b):
    ax2.annotate(name, (U[i,0]+0.01, U[i,1]+0.01), fontsize=8)
ax2.scatter(Vt.T[:,0], Vt.T[:,1], color=ACCENT1, marker='s', s=100, zorder=3, label='Projects')
for j, name in enumerate(projects_b):
    ax2.annotate(name, (Vt.T[j,0]+0.01, Vt.T[j,1]+0.01), fontsize=8, color=ACCENT1)
ax2.axhline(0, color='#ddd', linewidth=0.8); ax2.axvline(0, color='#ddd', linewidth=0.8)
ax2.set_xlabel('1st singular vector'); ax2.set_ylabel('2nd singular vector')
ax2.set_title('People and projects in the same\nlatent space (SVD)', fontsize=11)
ax2.legend(fontsize=9)

plt.suptitle('SVD of incidence matrix B: joint embedding of people and projects',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)
print('Hana (bridge) sits between the two clusters — visible only in SVD, not in P.')

In [ ]:
# ── Bipartite community detection via modularity matrix ───────────────────────
# Build the bipartite graph and use Louvain on the projected network
B_graph = nx.Graph()
B_graph.add_nodes_from(people_b,   bipartite=0)
B_graph.add_nodes_from(projects_b, bipartite=1)
for i, person in enumerate(people_b):
    for j, proj in enumerate(projects_b):
        if B[i,j]: B_graph.add_edge(person, proj, weight=float(B[i,j]))

# Project onto people with weights; apply spectral partition
G_p = nx.from_numpy_array(P_wt)
G_p = nx.relabel_nodes(G_p, {i: people_b[i] for i in range(m_b)})
L_p = nx.laplacian_matrix(G_p).toarray().astype(float)
_, evecs_p = eigh(L_p)
km_b = KMeans(n_clusters=2, n_init=20, random_state=0)
comm_labels = km_b.fit_predict(evecs_p[:, 1:3])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
pos_bip = nx.bipartite_layout(B_graph, people_b)
nx.draw_networkx(B_graph, pos_bip, ax=ax1,
                 node_color=[NODE_COLOR if nd in people_b else ACCENT1 for nd in B_graph.nodes()],
                 edge_color=EDGE_COLOR, node_size=350, font_size=7,
                 font_color='white', width=1.2)
ax1.set_title('Bipartite graph\n(blue=people, orange=projects)', fontsize=11)
ax1.axis('off')

pal2 = [NODE_COLOR, ACCENT1]
nx.draw_networkx(G_p, nx.spring_layout(G_p, seed=3), ax=ax2,
                 node_color=[pal2[l] for l in comm_labels],
                 edge_color=EDGE_COLOR, node_size=400,
                 font_size=8, font_color='white', font_weight='bold', width=1.5)
ax2.set_title('Spectral communities in projected\nperson-person network', fontsize=11)
ax2.axis('off')

plt.suptitle('Bipartite community detection: work-allocation silos', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='hypergraph'></a>
---
## Part 14: Hypergraph Laplacian: Group Interactions Beyond Pairwise Ties

> *A hyperedge can connect any number of vertices … Δ = D_v − H W D_e⁻¹ Hᵀ … When every hyperedge has exactly two members, this reduces to (1/2)(D−A), that is, half the ordinary graph Laplacian, since each edge's contribution is divided by |e|=2. The eigenvectors are identical and eigenvalues halved, so all qualitative structure is preserved … The D_e⁻¹ normalisation discounts large hyperedges: a company-wide all-hands contributes less per pair than a focused four-person working session.*

In [ ]:
people_h  = ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hana']
meetings  = ['Standup-A','Standup-B','Cross-func','Leadership','Allhands']
H = np.array([
    [1,0,1,1,1],
    [1,0,1,0,1],
    [1,0,0,0,1],
    [0,1,1,1,1],
    [0,1,0,0,1],
    [0,1,1,1,1],
    [0,1,0,1,1],
    [1,1,1,0,1],
], dtype=float)

hw = np.array([1,1,2,2,0.5])   # meeting weights
W_h = np.diag(hw)
D_e_inv = np.diag(1.0 / H.sum(axis=0))
vdeg = H @ (W_h @ np.ones(len(meetings)))
D_v  = np.diag(vdeg)
Delta = D_v - H @ W_h @ D_e_inv @ H.T

# Normalised hypergraph Laplacian
D_v_si = np.diag(1.0 / np.sqrt(vdeg))
Theta = D_v_si @ Delta @ D_v_si
evals_h, evecs_h = eigh(Theta)

print('Incidence matrix H (people × meetings):')
print(pd.DataFrame(H.astype(int), index=people_h, columns=meetings).to_string())
print(f'\nHypergraph Fiedler value: {evals_h[1]:.4f}')

In [ ]:
# ── Compare hypergraph vs pairwise projection spectrum ─────────────────────────
P_pp = H @ H.T; np.fill_diagonal(P_pp, 0)
# laplacian() here is the local helper defined in Part 0 (def laplacian(A): return np.diag(A.sum(axis=1)) - A).
# It is equivalent to scipy.sparse.csgraph.laplacian for the combinatorial case, but is NOT that function.
L_pp = laplacian(P_pp)
d_pp = P_pp.sum(axis=1)
D_pp_si = np.diag(np.where(d_pp > 0, 1.0/np.sqrt(d_pp), 0))
evals_pp = np.sort(np.linalg.eigvalsh(D_pp_si @ L_pp @ D_pp_si))
_, evecs_pp = eigh(D_pp_si @ L_pp @ D_pp_si)

nh = len(people_h)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

x = np.arange(nh); w = 0.35
ax1.bar(x-w/2, evals_h,  width=w, label='Hypergraph Δ',        color=NODE_COLOR)
ax1.bar(x+w/2, evals_pp, width=w, label='Pairwise projection',  color=ACCENT1)
ax1.set_xticks(x); ax1.set_xticklabels([f'λ{i+1}' for i in range(nh)])
ax1.set_title('Spectrum: hypergraph vs pairwise projection', fontsize=11)
ax1.legend(); ax1.axhline(0, color='black', linewidth=0.5)

fv_h  = evecs_h[:, 1]
fv_pp = evecs_pp[:, 1]
if np.dot(fv_h, fv_pp) < 0: fv_pp = -fv_pp
x2 = np.arange(nh)
ax2.barh(x2-0.2, fv_h,  height=0.35, label='Hypergraph',        color=NODE_COLOR)
ax2.barh(x2+0.2, fv_pp, height=0.35, label='Pairwise proj.',     color=ACCENT1)
ax2.set_yticks(x2); ax2.set_yticklabels(people_h, fontsize=9)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Fiedler vector per person', fontsize=11); ax2.legend()

plt.suptitle('Hypergraph Laplacian captures group-level structure lost in pairwise projection',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='multilayer'></a>
---
## Part 15: Multi-Layer Networks & the Supra-Laplacian

> *An organisation whose collaboration layer has high algebraic connectivity but whose trust layer has low algebraic connectivity is one where people work together without trusting each other, a structurally fragile condition … The supra-Laplacian ℒ = 𝒟 − 𝒜 generalises everything from the single-layer case.*

In [ ]:
n_ml = 8
A_collab = np.array([[0,1,1,0,1,0,0,0],[1,0,1,1,0,0,0,0],[1,1,0,1,0,1,0,0],
                     [0,1,1,0,0,0,1,0],[1,0,0,0,0,1,1,1],[0,0,1,0,1,0,1,0],
                     [0,0,0,1,1,1,0,1],[0,0,0,0,1,0,1,0]], dtype=float)
A_trust  = np.array([[0,1,0,0,0,0,0,0],[1,0,1,0,0,0,0,0],[0,1,0,1,0,0,0,0],
                     [0,0,1,0,0,0,0,0],[0,0,0,0,0,1,0,0],[0,0,0,0,1,0,1,1],
                     [0,0,0,0,0,1,0,1],[0,0,0,0,0,1,1,0]], dtype=float)

fv_c = np.sort(np.linalg.eigvalsh(laplacian(A_collab)))[1]
fv_t = np.sort(np.linalg.eigvalsh(laplacian(A_trust)))[1]
print(f'Collaboration λ₂: {fv_c:.4f}  ← well-connected')
print(f'Trust        λ₂: {fv_t:.4f}  ← fragile')
print('\nDiagnosis: collaboration is healthy; trust is splitting into two isolated cliques.')
print('This is one of the most common and actionable ONA findings.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pos_ml = nx.spring_layout(nx.from_numpy_array(A_collab), seed=1)

for ax, mat, title, color in [
    (axes[0], A_collab, f'Collaboration layer\nλ₂={fv_c:.3f} (cohesive)', ACCENT2),
    (axes[1], A_trust,  f'Trust layer\nλ₂={fv_t:.3f} (fragile)',          ACCENT3),
]:
    nx.draw_networkx(nx.from_numpy_array(mat), pos_ml, ax=ax,
                     node_color=color, edge_color=EDGE_COLOR, node_size=450,
                     font_color='white', font_weight='bold', width=2)
    ax.set_title(title, fontsize=11); ax.axis('off')

# Supra-Laplacian spectrum vs coupling strength
cw_vals = np.linspace(0.01, 3.0, 50)
fv_sup  = []
for cw in cw_vals:
    C = cw * np.eye(n_ml)
    sA = np.block([[A_collab, C],[C, A_trust]])
    sL = np.diag(sA.sum(axis=1)) - sA
    fv_sup.append(np.sort(np.linalg.eigvalsh(sL))[1])

axes[2].plot(cw_vals, fv_sup, color=NODE_COLOR, linewidth=2.5)
axes[2].axhline(fv_c, color=ACCENT2, linestyle='--', linewidth=1.5, label=f'Collab λ₂={fv_c:.3f}')
axes[2].axhline(fv_t, color=ACCENT3, linestyle='--', linewidth=1.5, label=f'Trust λ₂={fv_t:.3f}')
axes[2].set_xlabel('Inter-layer coupling'); axes[2].set_ylabel('Supra-Laplacian λ₂')
axes[2].set_title('Coupling strength vs\nsupra-Laplacian connectivity', fontsize=11)
axes[2].legend(fontsize=9)

plt.suptitle('Multi-layer network: collaboration + trust layers', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='temporal'></a>
---
## Part 16: Temporal Tracking: The Spectral Health Dashboard

> *Track the spectrum quarterly. A rising Fiedler value and widening spectral gap means increasing organisational cohesion. Falling values signal fragmentation, often visible in the spectrum before it shows up in engagement surveys or performance data.*

In [ ]:
def make_org(n, pw, pc, n_groups=3, seed=0):
    rng = np.random.default_rng(seed)
    sz  = [n//n_groups]*n_groups; sz[-1] += n - sum(sz)
    p   = [[pw if i==j else pc for j in range(n_groups)] for i in range(n_groups)]
    return nx.stochastic_block_model(sz, p, seed=int(rng.integers(1_000_000)))

quarters = list(range(1, 13))
fv_series, gap_series = [], []

for q in quarters:
    if   q <= 4:  pw, pc = 0.5, 0.15
    elif q <= 8:  pw, pc = 0.5, max(0.01, 0.15 - (q-4)*0.03)
    else:         pw, pc = 0.5, 0.03 + (q-8)*0.04

    Gt = make_org(30, pw, pc, seed=q)
    if nx.is_connected(Gt):
        fv_series.append(fiedler_value(Gt))
        gap_series.append(normalized_laplacian_spectrum(Gt)[1])
    else:
        fv_series.append(0.0); gap_series.append(0.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, series, ylabel in [
    (axes[0], fv_series,  'Fiedler value λ₂'),
    (axes[1], gap_series, 'Normalised spectral gap'),
]:
    ax.plot(quarters, series, color=NODE_COLOR, linewidth=2.5, marker='o', markersize=5)
    ax.axvspan(1, 4,  alpha=0.08, color=ACCENT2)
    ax.axvspan(4, 8,  alpha=0.08, color=ACCENT3)
    ax.axvspan(8, 12, alpha=0.08, color=NODE_COLOR)
    ax.axvline(4, color=ACCENT3, linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(8, color=ACCENT2, linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Quarter'); ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' over time', fontsize=11)
    ax.set_xticks(quarters)
    for xpos, label, color in [(2,'Healthy',ACCENT2),(6,'Fragmentation',ACCENT3),(10,'Intervention',NODE_COLOR)]:
        # Use axes-fraction coords for y so the label sits near the top regardless of data scale
        ax.text(xpos, 0.93, label, ha='center', fontsize=8, color=color,
                transform=ax.get_xaxis_transform())

plt.suptitle('Spectral dashboard: fragmentation visible before engagement surveys',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)

<a id='threshold'></a>
---
## Part 17: Data Quality: Threshold Sensitivity & Eigenvalue Stability

> *The choice of threshold for binarizing weighted data has a substantial effect on spectral results … Eigenvalue stability under threshold variation is a useful diagnostic: results that change dramatically when the threshold moves by a small amount should be treated with caution.*

In [ ]:
np.random.seed(55)
n_th = 25
W_raw = np.random.exponential(scale=2.0, size=(n_th, n_th))
W_raw = (W_raw + W_raw.T) / 2
np.fill_diagonal(W_raw, 0)
for grp in [range(0,8), range(8,17), range(17,25)]:
    for i in grp:
        for j in grp:
            if i != j: W_raw[i,j] *= 3

thresholds = np.linspace(0.1, 8.0, 60)
fiedler_th, n_comps = [], []

for t in thresholds:
    Ab = (W_raw >= t).astype(float); np.fill_diagonal(Ab, 0)
    Gt = nx.from_numpy_array(Ab)
    nc = nx.number_connected_components(Gt)
    n_comps.append(nc)
    fiedler_th.append(fiedler_value(Gt) if nc == 1 else np.nan)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(thresholds, fiedler_th, color=NODE_COLOR, linewidth=2)
fv_arr  = np.array(fiedler_th, dtype=float)
fv_diff = np.abs(np.diff(np.nan_to_num(fv_arr)))
unstable = fv_diff > np.nanpercentile(fv_diff, 75)
for i, u in enumerate(unstable):
    if u: ax1.axvspan(thresholds[i], thresholds[i+1], alpha=0.3, color=ACCENT3)
ax1.set_ylabel('Fiedler value λ₂')
ax1.set_title('Threshold sensitivity: red zones = eigenvalue unstable (avoid)', fontsize=11)

ax2.bar(thresholds, n_comps, width=thresholds[1]-thresholds[0],
        color=np.where(np.array(n_comps)==1, ACCENT2, ACCENT3))
ax2.axhline(1, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Binarization threshold'); ax2.set_ylabel('Components')
ax2.set_title('Connected components (green=1 = safe, red=fragmented)', fontsize=11)

plt.suptitle('Threshold sensitivity analysis for weighted ONA data', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close(fig)
print('Practical rule: choose a threshold inside a stable green band.')
print('Run this plot on your own data before trusting any spectral result.')

<a id='reference'></a>
---
## Part 18: Summary Reference Table

A quick-reference guide linking every mathematical object to its ONA interpretation and the essay section where it appears.

In [ ]:
from IPython.display import display

ref = pd.DataFrame([
    ('Adjacency matrix A',           'Part 1',  'Who is connected to whom. The network IS the matrix.'),
    ('Degree matrix D',              'Part 2',  'Diagonal: how many connections each person has.'),
    ('Laplacian L = D − A',          'Part 2',  'Measures local mismatch. Rows sum to zero.'),
    ('Fiedler value λ₂(L)',          'Part 4',  '#zero eigenvalues=#components; λ₂>0 iff connected; large=resilient; small=fragile.'),
    ('Fiedler vector',               'Part 4',  'Best two-way community split. Sort nodes by this vector.'),
    ('Laplacian as heat operator',   'Part 3',  'L·T = local heat mismatch. Eigenmodes = diffusion frequencies.'),
    ('Spectral clustering',          'Part 5',  'k-means on eigenvectors 2…k+1 → k informal communities.'),
    ('In-centrality (HITS auth)',    'Part 8',  'Directed: who receives attention from high-status nodes.'),
    ('Out-centrality (HITS hub)',    'Part 8',  'Directed: who sends attention to high-status nodes.'),
    ('PageRank',                     'Parts 8,9','Directed random-walk influence; handles asymmetric ties.'),
    ('Eigenvector centrality',       'Part 9',  'Undirected: influence = leading eigenvector of A.'),
    ('Betweenness centrality',       'Part 9',  'Fraction of shortest paths through a node: bottleneck signal.'),
    ('A² off-diagonal',             'Part 10',  'Number of shared contacts between each pair.'),
    ('A³ diagonal / 2',             'Part 10',  'Triangle count per node: clique density vs brokerage.'),
    ("Burt's constraint",           'Part 10',  'Low = structural holes / broker position between cliques.'),
    ('Spectral gap (norm. L)',       'Part 11',  'Information diffusion speed; small = echo chambers.'),
    ('Cospectral graphs',           'Part 12', 'Same spectrum ≠ same structure. Use within-network tracking only.'),
    ('Incidence matrix B',           'Part 13', 'People × artifacts membership matrix.'),
    ('P = B·Bᵀ  (raw)',             'Part 13', 'Co-membership count; over-weights large artifacts.'),
    ('P weighted (÷ size)',          'Part 13', 'Inverse-size normalisation; discounts mass-participation events.'),
    ('P Jaccard normalised',         'Part 13', 'Adjusts for activity breadth; comparable across people.'),
    ('SVD of B',                     'Part 13', 'Places people AND artifacts in the same latent space.'),
    ('Hypergraph Laplacian Δ',       'Part 14', 'Group-level connectivity; D_e⁻¹ discounts large meetings.'),
    ('Supra-adjacency matrix 𝒜',    'Part 15', 'nL×nL block matrix encoding all layers + coupling.'),
    ('Supra-Laplacian ℒ',           'Part 15', 'Multi-layer connectivity; reveals formal vs informal divergence.'),
    ('Spectral time series',         'Part 16', 'Rising λ₂ = cohesion; falling = fragmentation — leads surveys.'),
    ('Spectral graph drawing (Hall)',  'Part 6',  'v₂=x, v₃=y: mathematically optimal layout (Rayleigh-Ritz: v₂ minimises ∑edge lengths² subject to x⊥1).'),
    ('Higher-dim. spectral embedding', 'Part 6',  'v₂…v_{k+1} as coordinates — use k eigenvectors for k communities.'),
    ('λ₂ ↔ drawability theorem',       'Part 6',  'Large λ₂ → no clean drawing exists; high connectivity IS the finding.'),
    ('Threshold sensitivity plot',   'Part 17', 'Choose threshold in the stable zone; red zones = unreliable.'),
], columns=['Object', 'Essay part', 'ONA interpretation'])

display(
    ref.style
       .set_properties(**{'text-align': 'left', 'font-size': '12px'})
       .set_table_styles([{
           'selector': 'th',
           'props': [('background-color','#4C72B0'),
                     ('color','white'),
                     ('font-weight','bold'),
                     ('text-align','left')]
       }])
       .hide(axis='index')
)

---
*End of companion notebook v3. All examples use synthetic or public-domain data (Karate Club network). To apply to real ONA data: replace toy matrices with your adjacency, incidence, or interaction matrices and re-run each section.*

**File-size reminder:** If the notebook grows large after running, use **Edit → Clear All Outputs** before saving, or run `jupyter nbconvert --ClearOutputPreprocessor.enabled=True --to notebook your_file.ipynb --output your_file.ipynb` from the terminal.